In [3]:
import numpy as np
import matplotlib.pyplot as plt

In [280]:
##   -Code not yet tested... DONE
##    -test Lin_2_Opt_fast ... DONE


##TODO: 


##      -merge functions and give option flags for different outputs and memories instead
##      -write function that searches for least weighted path by comparing every single permutation (see C++ code)
##      - Abbruchkriterium: Nach wie vielen Schritten hören wir auf?
##      - Auswahl welche lin-Opt-methode genutzt werden soll
##      -Basis python-template (siehe unten)


def twoPointDist(x, y, dist_matrix): # funktioniert
    """ input:  two points x and y - (int)
                matrix giving the distances between points - (2d array)
        return: distance (int) between point x and point y (direction x to y)
    """
    return(dist_matrix[x, y])

def totalTravelDist(points, dist_matrix, loop=True): # korrigiert, funktioniert
    """ input:  points - array of points that gives the path - array
                dist_matrix - matrix containing the distances between all points (in both directions) - 2d array
                loop - info if the path needs to be a loop (start position = end position) - boolean
        output: dist_sum - total weight of the given path - int
    """
    N = len(points)
    if loop:
        dist_sum = twoPointDist(points[N-1], points[0], dist_matrix)
        for i in range(N-1):
            dist_sum += twoPointDist(points[i], points[i+1], dist_matrix)

    else:
        dist_sum = 0
        for i in range(N-1):
            dist_sum += twoPointDist(points[i], points[i+1], dist_matrix)
    return(dist_sum)

def travelWeightTotal_changingDist(points, distances, periodicity, loop = True): #korrigiert,funktioniert
    
    """ input:  points - array of points representing the path - array
                distances - matrices containing the distances between all points (in both directions) at different 'times' (3rd dim = time dim) - 3d array
                periodicity - periodicity of the time matrices in time - int
                loop - info if the path needs to be a loop (start position = end position) - boolean
        return: dist_sum - total weight of the given path - int
    """
    N = len(points)
    dist_sum = 0
    for i in range(N-1):
        dist_sum += twoPointDist(points[i], points[i+1], distances[:,:,round(dist_sum) % periodicity])
    if loop:
        dist_sum += twoPointDist(points[N-1], points[0], distances[:,:,round(dist_sum) % periodicity])
    return(dist_sum)

def exchange_two_parts(x1,x2,y1,y2,array):  # 
    
    """input: x1,x2 : beginning and end of first section
              y1,y2 : beginning and end of second section
              array : array which will get affected by the echange
        output:
                exchange the position [0,...,x1,...,x2,....,y1,...y2,...] to [0,...,y1,...,y2,....,x1,...x2,...]
                or vice-versa. x1 = x2 or y1 =y2 are possible
    """
    if x1 < y1:
        if x1 > 0:
            change = np.concatenate((array[0:x1],array[y1:y2+1], array[x2+1:y1], array[x1:x2+1], array[y2+1:]))
        else:
            change = np.concatenate((array[y1:y2+1], array[x2+1:y1], array[x1:x2+1], array[y2+1:]))
    else:
        if y1 > 0:
            change = np.concatenate((array[0:y1],array[x1:x2+1], array[y2+1:x1], array[y1:y2+1], array[x2+1:]))
        else:
            change = np.concatenate((array[x1:x2+1], array[y2+1:x1], array[y1:y2+1], array[x2+1:]))
    return(change)

##the following function should be enough in any case that provides ordered x1, x2, y1, y2 (smallest to largest)
def exchangeTwoPartsFast(x1, x2, y1, y2, arr): # funktioniert
    """ input:  x1, x2, y1, y2 - positions in array at which the array is to be cut in subarrays (it needs to be x1<x2<y1<y2) - int
                array - array which is to be reordered - array
        return: reordered array (exchange the position [0,...,x1,...,x2,....,y1,...y2,...] to [0,...,y1,...,y2,....,x1,...x2,...]) - array
    """
    return(np.concatenate((arr[0:x1],arr[y1:y2+1], arr[x2+1:y1], arr[x1:x2+1], arr[y2+1:])))

def Lin_3_Opt (way): 
    """
    Lin-3-Opt  exchanges two random successive parts of the tour without altering their directions.
    """
    candidates =np.array([])
    while len(candidates)==0:
        a1 = np.random.randint(len(way))
        a2 = np.random.randint(len(way))
        positions = np.arange(len(way))
        if a1 < a2:
            x1 = a1
            x2 = a2
        else:
            x1 = a2
            x2 = a1

        ind = (positions < x1) | (positions > x2)
        candidates = positions[ind]

    b1 = int(np.random.choice(candidates))
    
    if b1 < x1:
        b2 = int(np.random.choice(positions[:x1]))
    else: 
        b2 = int( np.random.choice(positions[x2+1:]))
        
    if b1 < b2:
            y1 = b1
            y2 = b2
    else:
            y1 = b2
            y2 = b1

    return way[exchange_two_parts(x1,x2,y1,y2,positions)]

def Lin_3_Opt_fast (path): # korrigiert, funktioniert
    """ input:  path - array that is to be reordered - array
        return: reordered path (exchanges two random successive parts of the tour without altering their directions) - array
    """
    ordered_cuts = np.array([0,0,0,0])
    while ordered_cuts[1]== ordered_cuts[2]:
        
        cuts = np.random.randint(len(path), size=4) ##take for random positions in path-array and put them in array
        ordered_cuts = np.sort(cuts)                #sort the array from smallest to largest
        
    return(exchangeTwoPartsFast(ordered_cuts[0],ordered_cuts[1],ordered_cuts[2],ordered_cuts[3],path)) 
    ##ordered intput so we can use exchangeTwoPartsFast
    
def Lin_2_Opt(way):
    """
    Lin-2-Opt move simply reverses the direction of a
    part of the tour  
    """
    a1 = 0
    a2 = 0
    while a1==a2:
        a1 = np.random.randint(len(way))
        a2 = np.random.randint(len(way))
    positions = np.arange(len(way)) 
    if a1 < a2:
        x1 = a1
        x2 = a2
    else:
        x1 = a2
        x2 = a1
    change = np.concatenate((positions[0:x1], positions[x1:x2+1][::-1], positions[x2+1:]))
    return way[change]

def Lin_2_Opt_fast(path):   #funktioniert so nicht_siehe Kommentare
    """ input:  path - array that is to be reordered - array
        return: reordered path (Lin-2-Opt move simply reverses the direction of a part of the tour ) - array
    """
    positions = np.arange(len(path)) 
    x2 = np.random.randint(len(path)) ##take random postion in path-array
    if x2 >0:
        x1 = np.random.randint(x2)  ##take random postion lower than previously selected position
    else: 
        x1 =0
    change = np.concatenate((positions[0:x1], positions[x1:x2+1][::-1], positions[x2+1:])) 
    ##make the change with the already ordered positions
    return(path[change])

def traveling_Yann(points, matrix, steps, T=1, alpha = 0.999):
    """
    Time independent traveling salesman with toggable temperature and temperature reduction (0.8<alpha<0.999)
    """
    
    cost = np.zeros(steps+1)
    temp = np.zeros(steps+1)
    C = np.zeros(steps+1)
    start = points
    summ = totalTravelDist(start, matrix)
    cost[0] = summ
    temp[0] = T
    C[0] = np.var(cost)/T**2
    
    for tries in range(steps):
        
        method = np.random.choice(2) # choose if you use Lin
        if method ==1:
            new_start = Lin_2_Opt(start)
        else:
            new_start = Lin_3_Opt(start)
        new_sum = totalTravelDist(new_start, matrix)
        if np.random.uniform(low=0, high=1) < np.exp((summ-new_sum)/T):
            start = new_start
            summ = totalTravelDist(new_start, matrix)
        cost[tries+1] = summ
        C[tries+1] = np.var(cost)/T**2
        temp[tries+1] = T
        T = alpha*T     
    return start, summ, cost, temp, C


### Time independent traveling salesman with toggable temperature and temperature reduction (0.8<alpha<0.999)
##  If we really want to trim it down and make it more storage efficient we could only get the best path and best weight 
# as output
def traveling_MC_constDist_Julius(points, dist_matrix, tries, starting_temp = 1, alpha = 0.999):#korrigiert,funktioniert
    """ input:  points - points that should be visited by each path (in some order) - array
                dist_matrix - matrix containing the 'distances' between each point - 2darray
                tries - itterations of the Monte Carlos - Markov chain - int
                starting_temp - scalar in the exponent of the markov chain acception process - int
                alpha - scalar refgulating the decrease of the temperature after each loop iteration in the MC
        return: points - points that should be visited by each path (in some order) - array
                pathWeights - 'distances' (better weights) of each path - array of int
                temps - temperatures after each MC step - array of int
    """
    
    pathWeights = np.zeros(tries+1)
    temps = np.zeros(tries+1)
    pathWeights[0] = totalTravelDist(points, dist_matrix)
    temps[0] = starting_temp
    cost = pathWeights[0]
    
    for i in range(tries):
        
        new_sequence = Lin_3_Opt(points)
        new_cost = totalTravelDist(new_sequence, dist_matrix)
        if np.random.uniform(low=0, high=1) < np.exp((pathWeights[i]-new_cost)/temps[i]):
            points = new_sequence
            cost = new_cost
        pathWeights[i+1] = cost
        #C[i+1] = np.var(cost)/T**2
        temps[i+1] = alpha*temps[i] 
    return(points, pathWeights, temps)

def traveling_MC_constDist_history_Julius(points, dist_matrix, tries, starting_temp = 1, alpha = 0.999):#korrigiert,funktioniert
    """ input:  points - points that should be visited by each path (in some order) - array
                dist_matrix - matrix containing the 'distances' between each point - 2darray
                tries - itterations of the Monte Carlos - Markov chain - int
                starting_temp - scalar in the exponent of the markov chain acception process - int
                alpha - scalar regulating the decrease of the temperature after each loop iteration in the MC
        return: pathWeights - 'distances' (better weights) of each path - array of int
                pathHistory - all paths in the order they have been explored (same as pathWeights) - 2darray
                temps - temperatures after each MC step - array of int
    """
    pathHistory = np.zeros((tries+1, len(points)))
    pathWeights = np.zeros(tries+1)
    temps = np.zeros(tries+1)
    #C = np.zeros(points+1)
    pathWeights[0] = totalTravelDist(points, dist_matrix)
    temps[0] = starting_temp
    cost = pathWeights[0]
    pathHistory[0] = points

    
    for i in range(tries):
        
        new_sequence = Lin_3_Opt(points)
        new_cost = totalTravelDist(new_sequence, dist_matrix)
        if np.random.uniform(low=0, high=1) < np.exp((pathWeights[i]-new_cost)/temps[i]): ##should it be temps[i+1]?????
            points = new_sequence
            cost = new_cost
        pathWeights[i+1] = cost
        pathHistory[i+1] = points
        temps[i+1] = alpha*temps[i]      
    return(pathWeights, pathHistory, temps)

###now time dependent with matrices sored in dim a (3darray 3rd dim is 'the' time)
def traveling_changingDist_MC(points, distances, tries, period, starting_temp = 1, alpha = 1 ):#korrigiert,funktioniert
    """
    Time dependent traveling salesman with toggable temperature and temperature reduction (0.8<alpha<0.999)
    """
    pathHistory = np.zeros((tries+1, len(points)))
    pathWeights = np.zeros(tries+1)
    temps = np.zeros(tries+1)
    #C = np.zeros(points+1)
    pathWeights[0] = travelWeightTotal_changingDist(points, distances, period)
    temps[0] = starting_temp
    cost = pathWeights[0]
    pathHistory[:][0] = points
    #C[0] = np.var(pathWeights)/T**2 ##What is this? Warum ist diese Groesse fuer uns von Bedeutung?
    
    for i in range(tries):
        
        new_sequence = Lin_3_Opt(points)
        new_cost = travelWeightTotal_changingDist(new_sequence, distances, period)
        if np.random.uniform(low=0, high=1) < np.exp((pathWeights[i]-new_cost)/temps[i]): ##should it be temps[i+1]?????
            points = new_sequence
            cost = new_cost
        pathWeights[i+1] = cost
        pathHistory[i+1] = points
        #C[tries+1] = np.var(cost)/T**2
        temps[i+1] = alpha*temps[i]   
    return(pathWeights, pathHistory, temps)



# Neu
Habe Zufallsmatrizenkreator gemacht mit speicherbaren Matrizen für spätere Vergleichstests

In [77]:
def create_testmatrix_2d(high, N, symmetric = True, low = 1, save ='0'): 
    """
    input: high: highest distance between two points - int or float
            N : number of points of interest -> dimension of matrix _int
            symmetric: default True, result will be symmetric matrix
            low: default 1, lowest possible distance _ int or float  
    output: random matrix for testpurpose
    """
    
    testmatrix = np.random.randint(low, high =high,size=(N,N))
    for i in range(N):
        testmatrix[i,i] = 0
    if symmetric:
        testmatrix = (testmatrix+testmatrix.T)/2
        np.savetxt('testmatrix_sym_('+str(low)+'_'+str(high)+')_'+str(save)+'.txt', testmatrix)
    else:
        np.savetxt('testmatrix_('+str(low)+'_'+str(high)+')_'+str(save)+'.txt', testmatrix)
    return testmatrix

def create_testmatrix_3d(high, N, time, symmetric = True, low = 1, save ='0'): 
    """
    input: high: highest distance between two points - int or float
            N : number of points of interest -> dimension of matrix _int
            time : number of different time matrices
            symmetric: default True, result will be symmetric matrix
            low: default 1, lowest possible distance _ int or float  
    output: random matrix for testpurpose
    """
    
    testmatrix = np.random.randint(low, high =high,size=(N,N,time))
    for j in range(time):
        for i in range(N):
            testmatrix[i,i,j] = 0
    if symmetric:
        for i in range(time):
            testmatrix[:,:,i] = (testmatrix[:,:,i]+testmatrix[:,:,i].T)/2
        with open('testmatrix_sym_('+str(low)+'_'+str(high)+')_t'+str(time)+'_'+str(save)+'.txt', 'w') as outfile:

            outfile.write('# Array shape: {0}\n'.format(testmatrix.shape))

            for data_slice in testmatrix:

                np.savetxt(outfile, data_slice, fmt='%-7.2f')

                # Writing out a break to indicate different slices...
                outfile.write('# New slice\n')
    else:
        with open('testmatrix_('+str(low)+'_'+str(high)+')_t'+str(time)+'_'+str(save)+'.txt', 'w') as outfile:

            outfile.write('# Array shape: {0}\n'.format(testmatrix.shape))

            for data_slice in testmatrix:

                np.savetxt(outfile, data_slice, fmt='%-7.2f')

                # Writing out a break to indicate different slices...
                outfile.write('# New slice\n')
    return testmatrix
        
def load_matrix(name): 
    """
        saves previously randomly generatet matrix    
        input: savename of testmatrix: 'testmatrix__('+str(low)+'_'+str(high)+')_'+str(save)+'.txt'
        output: testmatrix
    """
    data = np.loadtxt(name)
    return data
    


Möglicher Einbau in den MC, um die Lin Methode herauszufinden

In [ ]:
def Opt_method(x,array):
    """
    Chooses method of path change
    
    input: x - = 0,1 or 2: 0: Only Lin-2, 1: Only Lin-3, 2: randomize between both
        array - array to be alterad
    output: new array with either Lin-2 or Lin-3 used upon
    """
    if x == 0:
        return Lin_2_Opt_fast(array)
    elif x == 1:
        return Lin_3_Opt(array)
    else:
        method = np.random.choice(2) # choose if you use Lin
        if method ==1:
            return Lin_2_Opt_fast(array)
        else:
            return Lin_3_Opt(array)
    

# Korrekturen 

Hier meine Korrekturen in folgendem Schemata:

welche Funktion :  $\color{green}{\text{was ich korrigiert habe}}$ | wieso

$\text{totalTravelDist}$ : $\color{green}{\text{points[N] -> points[N-1]}}$ |

points[N-1] ist der letzte Eintrag von array der Länge N

$\text{travelWeightTotal_changingDis}t$: $\color{green}{\text{round(dist_sum) hinzugefügt}}$ |

% braucht integer -> Vorteil Zeit, Distanz kann nun problemlos Kommazahlen handhaben

$\text{Lin_3_Opt_fast}$ :  $\color{green}{\text{ordered_cuts = np.array([0,0,0,0])  und while ordered_cuts[1]== ordered_cuts[2] hinzugefügt}}$|

verhindert, dass wir zweimal dieselbe Stelle vertauschen -> np.concatenate verlängert sonst unser array z.B [1,2,3,4,4] statt [1,2,3,4] beim 'vertauschen' von 4 mit 4

$ \text{traveling_MC_constDist_Julius}$: $\color{green}{\text{np.zeros(points+1)-> np.zeros(tries+1)}}$ | Wir bekommen nach jeden tries ein neuen Wert und nicht für jeden Punkt

$\color{green}{\text{T -> temps[i]}}$ | Du nutzt statt T sowieso temps
                 
$ \text{traveling_MC_constDist_history_Julius}$: $\color{green}{\text{pathHistory = np.zeros((points, tries+1))->  pathHistory = np.zeros((tries+1, len(points)))}}$ | Wir bekommen nun eine Matrix mit der Punktgeschichte

$\color{green}{\text{np.zeros(points+1)-> np.zeros(tries+1)}}$ | Wir bekommen nach jeden tries ein neuen Wert und nicht für jeden Punkt

$\color{green}{\text{T -> temps[i]}}$ | Du nutzt statt T sowieso temps

$\text{traveling_changingDist_MC}$:$\color{green}{\text{pathHistory = np.zeros((points, tries+1))->  pathHistory = np.zeros((tries+1, len(points)))}}$ | Wir bekommen nun eine Matrix mit der Punktgeschichte

$\color{green}{\text{np.zeros(points+1)-> np.zeros(tries+1)}}$ | Wir bekommen nach jeden tries ein neuen Wert und nicht für jeden Punkt

$\color{green}{\text{T -> temps[i]}}$ | Du nutzt statt T sowieso temps

# Kommentare / Testergebnisse

## exchange_two_parts fast or not?
Ich habe mal die Zeit gemessen ob fast wirklich schneller ist. Habe dafür ein riesiges array genommen.

In [140]:
testpoints = np.linspace(0,10**5,10**5) 

In [141]:
%%timeit
exchange_two_parts(0,12321,223244,3424253,testpoints)

40.5 µs ± 2.21 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [143]:
%%timeit
exchangeTwoPartsFast(0,12321,223244,34242535,testpoints)

43 µs ± 3.47 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


Wie du siehst gibt es keinen zeitlichen Vorteil Fast zu nehmen. exchange_two_parts hat zwar mehr Code, aber da ist die Reihenfolge der Einträge egal.

## Lin_3_Opt_fast

 Wieseo Lin-3-Opt.Steht so im Paper ist anscheinend gebräuchlich bei tsp. Es heißt so weil 3 neue 'Kanten' entstehen. Lin steht wahrscheinlich für linear, weil wir lineare sequenzen tauschen.[0,...,x1-1, neu:y1,...y2,neu: x2+1,..y1-1, neu: x1...x2, y2+1,... ].
Dies habe ich Lin_3_opt_fast getestet. Dies ist tatsächlich (mit der Korrektur) schneller. Wir können also auf einer der beiden exchange_Funktionen verzichten.

In [201]:
testpoints = np.linspace(0,10**4,10**4) 


In [157]:
%%timeit
Lin_3_Opt(testpoints)

120 µs ± 4.05 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [156]:
%%timeit
Lin_3_Opt_fast(testpoints)

27 µs ± 1.83 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [163]:
# Macht exchange einen Unterschied?
def Lin_3_Opt_fast_2 (path):
    """ input:  path - array that is to be reordered - array
        return: reordered path (exchanges two random successive parts of the tour without altering their directions) - array
    """
    ordered_cuts = np.array([0,0,0,0])
    while ordered_cuts[1]== ordered_cuts[2]:
        cuts = np.random.randint(len(path), size=4) ##take for random positions in path-array and put them in array
        ordered_cuts = np.sort(cuts)                #sort the array from smallest to largest
    return(exchange_two_parts(ordered_cuts[0],ordered_cuts[1],ordered_cuts[2],ordered_cuts[3],path)) 

In [158]:
%%timeit
Lin_3_Opt_fast_2(testpoints)
# Antwort: Nein

27.2 µs ± 1 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


## Lin-2-Opt

Auf deiner Frage bezüglich habe ich folgenden Test durchgeführt: Wie oft wird die 75000-te Position in [1,...100000] getauscht?


In [200]:
def Lin_2_Opt_short_code(path):   #code-verkürzte Version 
    """ input:  path - array that is to be reordered - array
        return: reordered path (Lin-2-Opt move simply reverses the direction of a part of the tour ) - array
    """
    positions = np.arange(len(path)) 
    
    cut =  np.random.randint(len(path),size =2) ##take random postion in path-array
    order_cut = np.sort(cut)
    change = np.concatenate((positions[0:order_cut[0]], positions[order_cut[0]:order_cut[1]+1][::-1], positions[order_cut[1]+1:])) 
    ##make the change with the already ordered positions
    return(path[change])


points = np.arange(100000)
Yann_opt_2 = 0
Julius_opt_2 = 0
Yann_Julius_opt_2 = 0
for x in range(50000):
    if Lin_2_Opt(points)[75000] < 75000:
        Yann_opt_2 += 1
    if Lin_2_Opt_fast(points)[75000] < 75000:
        Julius_opt_2 += 1
    if Lin_2_Opt_short_code(points)[75000] < 75000:
        Yann_Julius_opt_2 += 1
print(Yann_opt_2,Julius_opt_2,Yann_Julius_opt_2)

15698 9008 15632


In [209]:
%%timeit
Lin_2_Opt(testpoints)

68.8 µs ± 1.04 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [210]:
%%timeit
Lin_2_Opt_short_code(testpoints)

76.5 µs ± 1.7 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


Deine Version hat wie befürchtet eher zu niedrigeren Werten geführt. Ich konnte aber eine code-verkürzte Funktion basierend auf deine Version kreieren. Von den Zeiten her scheint meine ursprüngliche Version schneller zu sein, aber nur marginal. Liegt wahrscheinlich an np.sort

## Deine MC

Ich habe deine MCs getestet und soweit korrigiert, dass sie funktioneren. Deine Ergebnisse stimmen auch soweit mit meinen überein (Nach mehreren durchgeführten Tests). Was noch optimierbar wäre, wäre vielleicht die Opt_methode einzufügen. Die Idee mit der Geschichtsspeicherung kann für später sehr nützlich werden.

Ich habe übrigens die festen alphas auf 0.999 gesetzt, damit bei (sehr) hohen tries es unwahrscheinlicher zu einer Änderung kommt. Siehe folgendes Beispiel.

Ps: Es gab verschiedene minimale Werte in dieser Matrix A

In [293]:
print(traveling_MC_constDist_Julius(points,A,100, alpha=1)[0:2])

(array([3, 1, 2, 0]), array([20.5, 20.5, 19. , 19. , 19. , 19. , 19. , 19. , 20.5, 20.5, 20.5,
       20.5, 20.5, 20.5, 20.5, 20.5, 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 20.5, 20.5, 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 20.5, 20.5, 20.5, 20.5, 20.5, 19. ,
       20.5, 20.5, 20.5, 20.5, 20.5, 20.5, 20.5, 20.5, 19. , 19. , 19. ,
       19. , 20.5, 20.5, 20.5, 20.5, 20.5, 20.5, 20.5, 20.5, 20.5, 20.5,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       20.5, 19. ]))


In [294]:
print(traveling_MC_constDist_Julius(points,A,100,alpha = 0.999)[0:2])

(array([2, 0, 3, 1]), array([20.5, 20.5, 20.5, 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 20.5, 20.5,
       20.5, 20.5, 20.5, 19. , 19. , 20.5, 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 20.5, 20.5, 20.5,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. ]))


In [296]:
print(traveling_MC_constDist_Julius(points,A,100,alpha = 0.9)[0:2])

(array([1, 3, 0, 2]), array([20.5, 20.5, 20.5, 20.5, 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. , 19. ,
       19. , 19. ]))


## Basis python-template

Ich würde vorschlagen ein python-template mit momentan folgenden Funktionen:

* twoPointDist
* totalTravelDist
* travelWeightTotal_changingDist
* exchangeTwoPartsFast, aber in exchangeTwoParts unbenannt
* Lin_3_Opt_fast, aber in Lin_3_Opt unbenannt
* Lin_2_Opt_short_code, aber in Lin_2_Opt unbenannt

Für die MC, würde ich deine nehmen, aber mit der Opt_method funktion implementiert. Außer es fällt dir eine andere Methode ein wie wir zwischen Lin_2 und Lin_3 wechseln